# 02 — The noise estimator, and what the quantiser does to it

Build step 2 (`DECISIONS` D16). One question: **how do we measure the spread of a set of
pixels on this camera?**

That sounds like it should be one line of numpy. It is not, and the reason is the subject of
this notebook. Notebook 01 established that every value this camera stores is an exact
multiple of 16 — a 12-bit ADC bit-shifted into a 16-bit file — and that on ~90% of the
calibration frames in the archive the *entire* measured spread is one such step. An estimator
running on those frames is not measuring the sensor, it is measuring the grid.

The output is `astropix/stats.py` and one file:

| file | cell | cost |
|---|---|---|
| `quantiser_bias.csv` | *The simulation* — estimator bias against a known truth | ~1 min |

Everything here runs locally except the last section, which reads two real flats from Z: to
show why the PTC differences frames rather than measuring one.

Decisions this notebook produced: **D30** (the two corrections and the declared floor) and
**D31** (`pair_variance` as the PTC's estimator).

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropix import cfa, fits as F, stats
from astropix import test as T          # bias_table lives with the tests (D23)

pd.set_option("display.width", 200)
RESULTS = pathlib.Path("..") / "results"

idx = pd.read_csv(RESULTS / "frame_index.csv")
ok = idx[idx.status == "ok"].copy()

STEP = stats.QUANTUM
print(f"quantiser step: {STEP:.0f} file-ADU   "
      f"(one ADC count; the 15 values between are unreachable)")
print(f"MAD can therefore only return multiples of "
      f"{stats.MAD_TO_SIGMA} x {STEP:.0f} = {stats.MAD_TO_SIGMA * STEP:.4f}")

## The failure, on real frames

MAD is the reflex choice for a robust spread: median of absolute deviations from the median,
scaled by 1.4826 to match a Gaussian's sigma. It is robust to stars and hot pixels, which is
exactly what a frame full of both requires.

But it is an **order statistic of the deviations**, and on this rig every deviation is a
multiple of 16. So MAD can only ever return a multiple of 23.7216 — it has no access to the
values between. Ask how often it lands on the very first rung.

In [ ]:
ONE_STEP = stats.MAD_TO_SIGMA * STEP

# bias and dark share a regime -- both sit at the pedestal -- so they are pooled.
# flats and lights are kept apart: lumping them would hide that they escape the
# grid for different reasons (a flat is bright by construction, a light by sky).
kind_col = np.where(ok.measured_type.isin(["bias", "dark"]),
                    "bias+dark", ok.measured_type)

rows = []
for (kind, gain), d in ok.assign(kind=kind_col).groupby(["kind", "gain"]):
    if len(d) < 100:
        continue
    rows.append({"frames": kind, "gain": gain, "n": len(d),
                 "at exactly one step": f"{(np.isclose(d.sigma, ONE_STEP)).mean():.1%}",
                 "median sigma": round(d.sigma.median(), 1)})
pd.DataFrame(rows)

**Nine tenths of every calibration frame in the archive, at both gains, reads out at exactly
one ADC step.** That number is a property of the quantiser and says nothing about the sensor.

The lights escape, and *why* they escape is the useful part: sky shot noise is large enough to
dither the quantiser for free. Rounding a value that is already jumping around by hundreds of
ADU scrambles the low bits instead of pinning them. That is the same mechanism that makes the
PTC work at all, and it is the warning that bias-frame read noise is the hardest number this
project has to produce.

## Why a correction curve cannot rescue MAD

A biased estimator is repairable — measure the bias, divide it out. So measure it: simulate
Gaussian noise at a known sigma, round it to the grid, and ask what MAD returns.

In [ ]:
for s in (0.25, 0.5, 1.0, 2.0, 4.0, 8.0):
    r = T.quantiser_bias(s, n=200_000, trials=4)
    print(f"true sigma = {s:>4} steps   MAD reads {r['mad']:.3f} x truth")

**1.483, then 0.741, then 1.112 — it is not monotonic.** MAD does not merely err, it cannot be
inverted: a reading of 23.72 is consistent with several different true sigmas, and no
correction curve can undo that. This is stronger than "MAD is biased here", and it settles
something practical — the `sigma` column of `frame_index.csv` can never be turned into a noise
measurement after the fact, not even with the table below in hand. It is a classification
feature (D24). Frames get re-measured, not rescaled.

Standard deviation has the opposite profile: it is nearly immune to the grid, because rounding
error averages over millions of pixels instead of being read off one order statistic — but it
is destroyed by a single hot pixel. Hence D24's estimator: **reject with a MAD-scaled cut,
which only needs to be roughly right, then take the std of the survivors.** Neither half
suffices alone.

## The rejection window has a floor, and it is not cosmetic

`clipped_std` cuts at `k x max(scale, quantum)`. Without that floor, a frame whose MAD comes
back **zero** — which happens whenever the noise is under about half a step, and which the
table above shows is the *normal* case for calibration frames here — would open a zero-width
window and reject every pixel that is not exactly at the median.

D24's "the cut only needs to be roughly right" is what makes the floor safe: a distribution
narrower than one step has no resolvable outliers to reject in the first place.

In [ ]:
flat_signal = np.full(10_000, 1232.0)          # no resolvable noise at all
raw, centre, kept = stats.clipped_std(flat_signal)
print(f"MAD of a constant array: {stats.MAD_TO_SIGMA * np.median(np.abs(flat_signal - 1232.0)):.1f}")
print(f"clipped_std keeps {kept:.0%} of pixels, centre {centre}, sigma {stats.sigma(flat_signal)}")
print("-> 0.0, not nan, and not 'all pixels rejected'")

## Correction 1 — Sheppard, and why it lands on the read noise

Rounding to a grid of step *q* behaves like adding an independent uniform error of width *q*.
A uniform distribution of width *q* has variance q²/12, and variances of independent things
add. So **every variance measured on this rig is inflated by a constant q²/12**, whatever the
gain, the exposure, or the brightness.

The consequence is the reason this notebook exists at all. A PTC fits variance against signal:
the slope is the gain, the intercept is the read noise. An **additive** offset moves the
intercept and leaves the slope untouched. So the quantiser biases `R(gain)` completely and
`g(gain)` not at all — and it is invisible at the bright end, which is exactly where you would
go looking for a problem.

In [ ]:
q2_12 = stats.sheppard_variance()
print(f"q^2/12 = {q2_12:.2f} file-ADU^2   (a constant, added to every variance)")
print()
print("Sheppard's prediction for an *uncorrected* std, against the simulation:")
print(f"{'sigma/step':>10} {'predicted':>10} {'measured':>10}")
for s in (0.5, 1.0, 2.0, 4.0, 8.0):
    pred = np.sqrt(s * s * STEP ** 2 + q2_12) / (s * STEP)
    meas = T.quantiser_bias(s, n=1_000_000, trials=4)["std_raw"]
    print(f"{s:>10.2f} {pred:>10.4f} {meas:>10.4f}")

Predicted and measured agree to the third decimal from half a step upward. This is not a
fitted fudge factor — it is arithmetic on a grid whose step we know exactly, confirmed against
a simulation that knows the truth. So the correction is a **subtraction**, not an error bar:
`sigma² = var_measured − q²/12`.

## Correction 2 — truncation, the one that is easy to forget

Clipping at ±k·sigma throws away the tails *of the distribution being measured*. The survivors
are therefore narrower than the parent, and their std reads low. For a Gaussian the shrinkage
is analytic, so there is no reason to tolerate it.

`clipped_std` divides it back out using the **effective** k the window actually reached, not
the nominal k. That matters because of the quantum floor above: when the floor is holding the
window open far out in the tails, nothing was truncated and there is nothing to correct.
Applying the nominal factor there would introduce the very bias it exists to remove.

In [ ]:
print(f"{'k':>5} {'var(survivors)/var(parent)':>28} {'std reads':>12}")
for k in (2.0, 3.0, 4.0, 5.0, 8.0):
    c = stats._truncation_factor(k)
    print(f"{k:>5.1f} {c:>28.5f} {np.sqrt(c) - 1:>11.2%}")
print()
print("with the grid effectively absent, the estimator must recover the truth:")
x = np.random.default_rng(7).normal(0.0, 100.0, 2_000_000)
print(f"  true 100.0  ->  {stats.sigma(x, quantum=1e-9):.3f}")

## The simulation

Both corrections in place, sweep the true sigma and ask what the estimator returns. `phase` is
not a detail: below one step the rounding stops being random and becomes a deterministic
function of the signal, so *where the true mean sits relative to the grid* changes the answer.
The measured pedestals on this rig are 1040.0 and 1232.0 — both **exact grid points**, the
least forgiving case. `"grid"` simulates that; `"random"` averages over it.

In [ ]:
rows = T.bias_table(n=1_000_000, trials=8)
bias = pd.DataFrame(rows)
bias.to_csv(RESULTS / "quantiser_bias.csv", index=False)
print(f"{len(bias)} rows -> quantiser_bias.csv")
bias[bias.phase == "grid"][["sigma_steps", "sigma_file_adu", "mad", "std_raw", "sigma", "pair"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
g = bias[bias.phase == "grid"]
for col, lbl, style in [("mad", "MAD (D24 rejected this)", ":"),
                        ("std_raw", "clipped std, uncorrected", "--"),
                        ("sigma", "stats.sigma  (both corrections)", "-"),
                        ("pair", "stats.pair_sigma", "-.")]:
    ax.plot(g.sigma_steps, g[col], style, marker="o", ms=4, label=lbl)
ax.axhline(1.0, c="k", lw=.8)
ax.axvspan(0.1, 0.5, color="0.9", zorder=0)
ax.text(0.16, 0.15, "unresolvable", fontsize=9, color="0.4")
ax.set_xscale("log")
ax.set_xlabel("true sigma  (quantiser steps of 16 file-ADU)")
ax.set_ylabel("estimated / true")
ax.set_title("what the grid does to four estimators")
ax.legend(fontsize=9)
plt.show()

## The floor, stated rather than hoped for

- **at and above one step** — exact to three decimals
- **at half a step** — 1.6% low in grid phase
- **at a quarter step** — every pixel rounds to the same value and the information is *gone*.
  `stats.sigma` returns **0.0** there, which is the design: a frame whose noise the quantiser
  cannot resolve must not hand a fit a plausible number. Callers exclude on it; they do not
  extrapolate through it.

So single-frame read noise is measurable down to **8 file-ADU (0.5 ADC counts)** and no
further. Below that, `R(gain)` comes from the PTC intercept — which is the next notebook, and
which is why the intercept had to be understood before it was fit.

## Why the PTC differences frames — on two real flats

`stats.pair_variance` is `var(a − b)/2` over two frames at identical settings. The argument for
it is not statistical elegance, it is that a single frame's spatial spread **cannot separate**
shot noise from fixed pattern: pixel-response non-uniformity, amp glow, dust motes on the
sensor window. All of those are the same in both frames, so differencing removes them and
leaves only what changed between two reads.

A flat is where this is most visible, because a flat is *made* of fixed pattern. Two real ones
from the archive, on a single CFA sub-plane (D4 — never debayer before a noise statistic):

In [ ]:
flats = ok[(ok.measured_type == "flat") & (ok.gain == 50.0) & (ok.exptime == 3.0)]
flats = flats.assign(folder=flats.path.str.rsplit("\\", n=1).str[0])
folder = flats.folder.value_counts().index[0]
pair = flats[flats.folder == folder].sort_values("path").path.iloc[:2].tolist()

a, _ = F.read(pair[0])
b, _ = F.read(pair[1])
print(pathlib.Path(pair[0]).name, "\n" + pathlib.Path(pair[1]).name)

pa = cfa.split(a)["R"].astype(np.float64)
pb = cfa.split(b)["R"].astype(np.float64)
print(f"\nR sub-plane {pa.shape}, level {np.median(pa):.0f} file-ADU")

spatial = stats.sigma(pa)
temporal = stats.pair_sigma(pa, pb)
print(f"\n  spatial sigma, one frame : {spatial:8.1f} file-ADU")
print(f"  temporal sigma, the pair : {temporal:8.1f} file-ADU")
print(f"  ratio                    : {spatial / temporal:8.2f}")
print(f"\n  implied fixed pattern    : {np.sqrt(max(spatial**2 - temporal**2, 0)):8.1f} "
      f"file-ADU = {np.sqrt(max(spatial**2 - temporal**2, 0)) / np.median(pa):.2%} of level")

**The single-frame number is wrong by the ratio above, and nothing about the curve would say
so.** Fixed pattern inflates the variance at every signal level, which steepens the
variance-vs-signal slope, and the gain is one over that slope — so `g` comes out **too low**,
and every electron count derived from it too small. A plausible-looking PTC, a wrong constant,
no warning anywhere. This is the same failure shape as
MAD: an estimator that never raises, and simply returns the wrong number.

The residual after differencing is the fixed pattern, and as a fraction of level it is the
PRNU — a real property of this sensor, which `03` will measure properly rather than infer from
one pair.

---

## Where this leaves the build

`stats.py` is 140 lines and exports four things: `sigma` for a single plane, `pair_variance`
and `pair_sigma` for the PTC, and `clipped_std` underneath them. Its error budget is on paper
rather than assumed, and it declares where it stops working.

Two things it does *not* yet have:

- **PixInsight contract 1** — our sigma against PI's noise evaluation on the same frame (D6).
  That needs `pixinsight.py`, which is build step 5, so it is deferred rather than skipped.
- **a real read-noise number.** Nothing here measured the sensor; it measured the estimator.

Next is **03**: the PTC over 61 gain steps, which settles the e-/ADU unit question that
`FINDINGS` calls the highest-priority open item, and produces the first two provenanced
constants — `g(gain)` and `R(gain)`.